# Full-image NPR with native sliding patches

Notebook này không resize ảnh. Một cửa sổ patch trượt với overlap trên toàn ảnh; NPR được tính riêng trên từng patch rồi cộng và chia cho coverage count để ghép lại thành NPR map toàn ảnh. FFT được tính trên full NPR map sau khi ghép.

```text
native image → overlapping sliding patches → NPR từng patch → average overlap → full NPR → FFT(full NPR)
```


In [ ]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F

DATA_ROOT_CANDIDATES = [Path('/kaggle/input/datasets/yangsangtai/tiny-genimage'), Path('/kaggle/input/tiny-genimage')]
PATCH_SIZES = (256, 128, 64, 32)
STRIDE_RATIO = 0.5
NUM_VISUAL_PER_GENERATOR = 2
NUM_VISUAL_PAIRS = 14
RANDOM_SEED = 42
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
rng = random.Random(RANDOM_SEED)

def get_image_paths(folder):
    return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS) if folder.exists() else []

def is_generator_dir(path):
    return path.is_dir() and (path / 'val' / 'nature').exists() and (path / 'val' / 'ai').exists()

def find_data_root():
    for root in DATA_ROOT_CANDIDATES:
        if root.exists() and any(is_generator_dir(p) for p in root.iterdir()):
            return root
    for root in Path('/kaggle/input').rglob('*'):
        if root.is_dir() and any(is_generator_dir(p) for p in root.iterdir()):
            return root
    raise FileNotFoundError('Attach yangsangtai/tiny-genimage.')

DATA_ROOT = find_data_root()
generator_dirs = sorted((p for p in DATA_ROOT.iterdir() if is_generator_dir(p)), key=lambda p: p.name)
print('Dataset root:', DATA_ROOT, '| generators:', len(generator_dirs))


In [ ]:
def sample_paths(paths, count):
    return list(paths) if len(paths) <= count else rng.sample(paths, count)

visual_pairs = []
for generator_dir in generator_dirs:
    real = get_image_paths(generator_dir / 'val' / 'nature')
    fake = get_image_paths(generator_dir / 'val' / 'ai')
    count = min(NUM_VISUAL_PER_GENERATOR, len(real), len(fake))
    for real_path, fake_path in zip(sample_paths(real, count), sample_paths(fake, count)):
        visual_pairs.append({'source': generator_dir.name, 'real_path': real_path, 'fake_path': fake_path})
rng.shuffle(visual_pairs)
visual_pairs = visual_pairs[:NUM_VISUAL_PAIRS]
print('Visual pairs:', len(visual_pairs))


In [ ]:
def choose_patch_size(short_side):
    return next((p for p in PATCH_SIZES if p <= short_side), None)

def sliding_starts(length, patch_size, stride):
    if length < patch_size:
        return []
    starts = list(range(0, length - patch_size + 1, stride))
    last = length - patch_size
    if last not in starts:
        starts.append(last)
    return starts

def patch_to_tensor(patch):
    rgb = np.asarray(patch, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(rgb).permute(2, 0, 1)
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD

def compute_npr_patch(patch_tensor):
    x = patch_tensor.unsqueeze(0)
    down = F.interpolate(x, scale_factor=0.5, mode='nearest', recompute_scale_factor=True)
    up = F.interpolate(down, size=x.shape[-2:], mode='nearest')
    return ((x - up) * (2.0 / 3.0)).squeeze(0)

def sliding_patch_npr(image, patch_size, stride_ratio=STRIDE_RATIO):
    # Crop tối đa một hàng/cột để H,W chẵn; không nội suy pixel.
    width, height = image.size
    width_even, height_even = width - width % 2, height - height % 2
    image = image.crop((0, 0, width_even, height_even)).convert('RGB')
    if min(image.size) < patch_size:
        raise ValueError(f'Image {image.size} smaller than patch {patch_size}.')
    stride = max(2, int(patch_size * stride_ratio))
    stride -= stride % 2
    x_starts = sliding_starts(image.width, patch_size, stride)
    y_starts = sliding_starts(image.height, patch_size, stride)
    residual_sum = torch.zeros(3, image.height, image.width)
    coverage = torch.zeros(1, image.height, image.width)
    boxes = []
    for top in y_starts:
        for left in x_starts:
            patch = image.crop((left, top, left + patch_size, top + patch_size))
            residual = compute_npr_patch(patch_to_tensor(patch))
            residual_sum[:, top:top + patch_size, left:left + patch_size] += residual
            coverage[:, top:top + patch_size, left:left + patch_size] += 1
            boxes.append((left, top, patch_size, patch_size))
    if torch.any(coverage == 0):
        raise RuntimeError('Sliding patches did not cover the full image.')
    return image, residual_sum / coverage, coverage.squeeze(0).numpy(), boxes

def npr_magnitude(npr):
    return torch.sqrt(torch.mean(npr.float() ** 2, dim=0)).numpy()

def compute_npr_psd(npr):
    residual = npr.numpy().astype(np.float64)
    height, width = residual.shape[-2:]
    window = np.outer(np.hanning(height), np.hanning(width))
    spectrum = np.fft.fftshift(np.fft.fft2(residual * window[None], axes=(-2, -1)), axes=(-2, -1))
    power = np.sum(np.abs(spectrum) ** 2, axis=0)
    power /= power.sum() + np.finfo(np.float64).eps
    return power, np.log10(power + 1e-12)

def radial_psd_statistics(power, bins=96):
    height, width = power.shape
    yy, xx = np.indices(power.shape)
    radius = np.sqrt((yy - height // 2) ** 2 + (xx - width // 2) ** 2) / (min(height, width) / 2.0)
    edges = np.linspace(0.0, 1.0, bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    density = np.full(bins, np.nan)
    band_energy = np.zeros(bins)
    valid_total = power[radius <= 1.0].sum() + np.finfo(np.float64).eps
    for index in range(bins):
        mask = (radius >= edges[index]) & (radius < edges[index + 1])
        if np.any(mask):
            density[index] = power[mask].mean()
            band_energy[index] = power[mask].sum() / valid_total
    density /= np.nansum(density) + np.finfo(np.float64).eps
    cumulative_energy = np.cumsum(band_energy)
    return centers, density, band_energy, cumulative_energy


In [ ]:
rows = []
for pair in visual_pairs:
    real_image = Image.open(pair['real_path']).convert('RGB')
    fake_image = Image.open(pair['fake_path']).convert('RGB')
    patch_size = choose_patch_size(min(min(real_image.size), min(fake_image.size)))
    if patch_size is None:
        continue
    row = {'source': pair['source'], 'patch_size': patch_size}
    for label, image in [('real', real_image), ('fake', fake_image)]:
        native_image, full_npr, coverage, boxes = sliding_patch_npr(image, patch_size)
        psd, log_psd = compute_npr_psd(full_npr)
        row.update({f'{label}_image': native_image, f'{label}_size': image.size, f'{label}_npr': npr_magnitude(full_npr), f'{label}_psd': psd, f'{label}_fft': log_psd, f'{label}_coverage': coverage, f'{label}_boxes': boxes})
    rows.append(row)

all_npr = np.concatenate([row[f'{label}_npr'].ravel() for row in rows for label in ('real', 'fake')])
all_fft = np.concatenate([row[f'{label}_fft'].ravel() for row in rows for label in ('real', 'fake')])
npr_vmax = np.percentile(all_npr, 99.5)
fft_vmin, fft_vmax = np.percentile(all_fft, [2.0, 99.5])
fig, axes = plt.subplots(len(rows), 6, figsize=(21, max(4, 3.5 * len(rows))), squeeze=False)
for row_index, row in enumerate(rows):
    for label, start in [('real', 0), ('fake', 3)]:
        axes[row_index, start].imshow(row[f'{label}_image'])
        for left, top, width, height in row[f'{label}_boxes']:
            axes[row_index, start].add_patch(Rectangle((left, top), width, height, fill=False, edgecolor='cyan', linewidth=0.45, alpha=0.45))
        axes[row_index, start].set_title(f"{label.upper()} — {row['source']}\noriginal={row[f'{label}_size']} | patch={row['patch_size']} | n={len(row[f'{label}_boxes'])}")
        axes[row_index, start + 1].imshow(row[f'{label}_npr'], cmap='gray', vmin=0, vmax=npr_vmax)
        axes[row_index, start + 1].set_title(f'{label.upper()} full stitched NPR')
        axes[row_index, start + 2].imshow(row[f'{label}_fft'], cmap='magma', vmin=fft_vmin, vmax=fft_vmax)
        axes[row_index, start + 2].set_title(f'{label.upper()} FFT(full NPR)')
    for axis in axes[row_index]:
        axis.axis('off')
fig.suptitle('Sliding native patches cover the full image — no resize', y=1.001)
plt.tight_layout()
plt.show()


## Lưu ý

- Mỗi cặp real–fake dùng cùng patch size; stride mặc định bằng nửa patch để có overlap.
- Các residual tại vùng overlap được average. Patch origin và stride đều chẵn để giữ cùng phase của lưới NPR 2×2.
- Hình NPR có cùng kích thước native với ảnh (trừ tối đa một hàng/cột lẻ), không phải một center patch.
- `imshow` chỉ scale khi render trên màn hình; dữ liệu NPR và FFT không bị resize.
- Có thể giảm `NUM_VISUAL_PAIRS` nếu figure quá lớn hoặc tăng `STRIDE_RATIO` lên `1.0` để dùng patch không overlap.
